# Baseline using matrix factorization

We use an implicit ALS model as our collaborative filtering baseline due to its efficiency on CPU and strong performance in implicit-feedback settings. It is used as the industry standard in many collabrative filtering settings Pairwise ranking approaches such as BPR were considered but found computationally expensive.

Note: implicit needs extra tools before it can be installed on PC, which is why I moved to colab.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
!pip install implicit

In [3]:
from google.colab import drive
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from implicit.als import AlternatingLeastSquares
import pickle
from collections import Counter
from torch.utils.data import Dataset,DataLoader
from implicit.evaluation import mean_average_precision_at_k
import random
import scipy.sparse as sp
np.random.seed(44)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import datasets

In [47]:

import os
import pickle
from pathlib import Path

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# 1. Define your base path relative to your Google Drive structure
# Replace 'Your_Project_Folder' with the actual folder name in your Drive
base_path = Path('/content/drive/MyDrive/project dl/datasets/processed')

# 2. Define paths using Path objects (handles / vs \ automatically)
train_sessions_path = base_path / 'train' / 'session_train.pkl'
val_sessions_path = base_path / 'val' / 'session_val.pkl'
test_sessions_path = base_path / 'val' / 'session_val.pkl'
# 3. Load the files
# Added a quick check to see if the files exist to avoid "File Not Found" errors
if train_sessions_path.exists():
    train_sessions = load_pkl(train_sessions_path)
    val_sessions = load_pkl(val_sessions_path)
    test_sessions= load_pkl(test_sessions_path)
    #train_360k = load_pkl(train_360k_path)
    #val_360k = load_pkl(val_360k_path)
    print("All datasets loaded successfully from Google Drive.")
else:
    print(f"Error: Could not find files at {base_path}. Please check your Drive folder name.")


All datasets loaded successfully from Google Drive.


In [48]:
with open(base_path /"track_vocab.pkl", "rb") as f:
    track_vocab = pickle.load(f)
with open(base_path /"artist_vocab.pkl", "rb") as f:
    artist_vocab = pickle.load(f)
with open(base_path /"user_vocab.pkl", "rb") as f:
    user_vocab = pickle.load(f)

# 1k dataset

In [49]:
train_sessions

,user_key,artist_key,track_key,session_start
global_session_id,,,,
user_000001_1,user_000001,"[6684, 6684, 6684, 72320, 20154, 74811, 74811,...","[737481, 706424, 640305, 817359, 67722, 448287...",2006-08-13 13:59:20+00:00
user_000001_10,user_000001,"[137499, 77373, 134399, 170765, 151205]","[441013, 460977, 198341, 434974, 240381]",2006-08-21 17:55:52+00:00
user_000001_100,user_000001,"[168162, 168162, 19742, 19742, 19742, 19742, 1...","[155170, 744218, 226370, 527575, 536705, 28745...",2006-11-16 01:49:47+00:00
user_000001_1000,user_000001,"[16883, 16883, 16883, 16883, 16883, 16883, 168...","[90123, 215003, 168408, 283808, 841571, 90123,...",2009-04-05 14:52:39+00:00
user_000001_1001,user_000001,"[16883, 16883, 16883, 16883, 16883, 16883, 168...","[652847, 840262, 107907, 90123, 215003, 840262...",2009-04-06 17:38:45+00:00
...,...,...,...,...
user_001000_986,user_001000,"[116163, 116163, 116163, 116163, 116163, 11616...","[862710, 612894, 539210, 928710, 430646, 96035...",2009-04-30 05:34:00+00:00
user_001000_988,user_001000,"[116163, 116163, 116163, 116163, 116163, 11616...","[768344, 187367, 291132, 150328, 105437, 32323...",2009-05-01 05:24:10+00:00
user_001000_989,user_001000,"[14447, 14447, 14447, 14447, 14447, 14447]","[332276, 635491, 507120, 552273, 805656, 115687]",2009-05-01 21:26:52+00:00


## Baseline 1: Random

In [7]:
def random_suggestions(train_df=train_sessions,k=10):
  all_tracks = df["track_key"].explode().unique()
  track_counts = all_tracks.value_counts()
  probs = track_counts / track_counts.sum()

  random_tracks = np.random.choice(
    probs.index,
    size=10,
    replace=False,
    p=probs.values
)
  return(random_tracks)


## Baseline 2: popularity suggestion

In [50]:
def popular_suggestions(train_df=train_sessions,k=10):
  popular_tracks = (
    train_df["track_key"]
    .explode()
    .value_counts()
  )
  top_k = popular_tracks.head(10)
  return(top_k)

## Baseline 3: Matrix factorization

In [66]:
df = train_sessions.explode("track_key")
df = df.groupby(["user_key", "track_key"]).size().reset_index(name="count")
users = df["user_key"].unique()
tracks = df["track_key"].unique()

user_to_idx = {u: i for i, u in enumerate(users)}
track_to_idx = {t: i for i, t in enumerate(tracks)}
idx_to_track = {i: u for u, i in track_to_idx.items()}
idx_to_user = {i: u for u, i in user_to_idx.items()}

In [67]:


rows = df["user_key"].map(user_to_idx).values
cols = df["track_key"].map(track_to_idx).values

data = np.log1p(df["count"].values)

user_track_matrix = sp.csr_matrix(
    (data, (rows, cols)),
    shape=(len(users), len(tracks))
).tocsr()

In [55]:
model =AlternatingLeastSquares(
                factors=150, regularization=0.01, iterations=20
            )
model.fit(user_track_matrix)

  0%|          | 0/20 [00:00<?, ?it/s]

In [57]:
val_sessions = val_sessions.copy()
val_sessions = val_sessions[val_sessions["track_key"].apply(len) >= 2]

In [56]:
def evaluate_model(model, val_sessions, track_to_idx, idx_to_track, K=10):

    hits = 0
    total = 0

    for user,session in val_sessions[["user_key","track_key"]].values:
        if user not in user_to_idx:
          continue
        prefix = session[:-1]
        target = session[-1]

        cols = [track_to_idx[t] for t in prefix if t in track_to_idx]

        session_prefix_vec = sp.csr_matrix(
            (np.ones(len(cols)), ([0] * len(cols), cols)),
            shape=(1, len(track_to_idx))
        )
        user_id = user_to_idx[user]

        user_vec = user_track_matrix[user_id].copy()

        user_vec = user_vec + session_prefix_vec
        #user_vec = 0.7 * user_vec + 0.3 * session_prefix_vec

        recs, scores = model.recommend(user_id, user_vec,
                                               recalculate_user=True,
                                               N=10,
                                      filter_already_liked_items=True)


        target_idx = track_to_idx.get(target, None)
        if target_idx is not None and target_idx in recs:
            hits += 1

        total += 1

    return hits / total

In [12]:
item_factors = final_model.item_factors
print(np.linalg.norm(item_factors, axis=1)[:10])

[0.293469   0.31081942 0.37780407 0.42309204 0.1647353  0.10381573
 0.23125537 0.3727195  1.6813653  0.34907338]


In [58]:


ranks = [50, 100, 150, 200]
regs = [0.005, 0.01, 0.05, 0.1]

best_score = 0
best_params = {}
for i in range(20):

    rank = random.choice(ranks)
    reg = random.choice(regs)

    model = AlternatingLeastSquares(
        factors=rank,
        regularization=reg,
        iterations=20
    )

    model.fit(user_track_matrix)

    score = evaluate_model(
        model,
        val_sessions,
        track_to_idx,
        idx_to_track,
        K=10
    )

    print(f"rank={rank}, reg={reg} -> Hit@10={score:.4f}")

    if score > best_score:
        best_score = score
        best_params = {"factors": rank, "reg": reg}

print(f"\nBest Params found: {best_params} with Hit@10: {best_score:.4f}")

  0%|          | 0/20 [00:00<?, ?it/s]

rank=200, reg=0.01 -> Hit@10=0.0021


  0%|          | 0/20 [00:00<?, ?it/s]

rank=50, reg=0.1 -> Hit@10=0.0031


  0%|          | 0/20 [00:00<?, ?it/s]

...

rank=50, reg=0.005 -> Hit@10=0.0031


  0%|          | 0/20 [00:00<?, ?it/s]

rank=50, reg=0.005 -> Hit@10=0.0031


  0%|          | 0/20 [00:00<?, ?it/s]

rank=200, reg=0.01 -> Hit@10=0.0000

Best Params found: {'factors': 50, 'reg': 0.1} with Hit@10: 0.0031


In [68]:
final_model =AlternatingLeastSquares(
                factors=50, regularization=0.01, iterations=20
            )
training = user_track_matrix.copy()
final_model.fit(training)

  0%|          | 0/20 [00:00<?, ?it/s]

In [61]:
evaluate_model(
        final_model,
        test_sessions,
        track_to_idx,
        idx_to_track,
        K=10
    )

0.003076923076923077

In [69]:
def give_recommendations(model, user,session_history, track_to_idx,idx_to_track,user_to_idx,  k=10):
  cols = [track_to_idx[t] for t in session_history if t in track_to_idx]

  session_prefix_vec = sp.csr_matrix(
        (np.ones(len(cols)), ([0] * len(cols), cols)),
        shape=(1, len(track_to_idx))
    )
  if user not in user_to_idx:
        recs, scores = model.recommend(0,session_prefix_vec,
                                               recalculate_user=True,
                                               N=k,
                                      filter_already_liked_items=True)
        target_idx = track_to_idx.get(target, None)
  else:
        user_id = user_to_idx[user]

        user_vec = user_track_matrix[user_id].copy()

        #user_vec = user_vec + session_prefix_vec
        user_vec = 0.7 * user_vec + 0.3 * session_prefix_vec

        recs, scores = model.recommend(user_id, user_vec,
                                               recalculate_user=True,
                                               N=k,
                                      filter_already_liked_items=True)


        rec_keys=[idx_to_track[r] for r in recs]
  return(rec_keys)


In [70]:
#example:
session=test_sessions.iloc[1]
user=session["user_key"]
prefix = session["track_key"][:-1]
target = session["track_key"][-1]
track_list=session["track_key"]
recs=give_recommendations(final_model,user,prefix,track_to_idx,idx_to_track,user_to_idx)
recs

[np.int64(68469),
 np.int64(210575),
 np.int64(789115),
 np.int64(130931),
 np.int64(244857),
 np.int64(270892),
 np.int64(834497),
 np.int64(876365),
 np.int64(196847),
 np.int64(933136)]

In [71]:
data_bundle = {
    "model": final_model,
    "user_to_idx": user_to_idx,
    "track_to_idx": track_to_idx,
    "idx_to_track": idx_to_track,
}

with open("recommender.pkl", "wb") as f:
    pickle.dump(data_bundle, f)